# 3.1 — Confusion Matrix

The confusion matrix breaks your predictions into 4 categories. Every other metric (precision, recall, F1) is calculated from these 4 numbers.

> **Why accuracy alone fails:** A model that predicts 'not cancer' for everyone scores 98% accuracy when only 2% have cancer — and catches zero real cases.

### The 4 categories:
- **TP** — True Positive: predicted YES, actually YES ✓
- **TN** — True Negative: predicted NO, actually NO ✓
- **FP** — False Positive: predicted YES, actually NO ✗ (false alarm)
- **FN** — False Negative: predicted NO, actually YES ✗ (missed case)

Memory trick: first word (True/False) = were you right? Second word (Positive/Negative) = what did you predict?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

# Load and prepare Titanic — same pipeline as chapter 2
df = sns.load_dataset('titanic')
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

X = df.drop(columns=['survived'])
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print("Model trained. Accuracy:", round(accuracy_score(y_test, y_pred), 3))

## Why Accuracy Fails

In [ ]:
# Class distribution
print("Class distribution in test set:")
print(y_test.value_counts())

# Dumb baseline — predict majority class for everyone
majority = y_test.mode()[0]
dumb_pred = [majority] * len(y_test)
dumb_acc = accuracy_score(y_test, dumb_pred)
our_acc  = accuracy_score(y_test, y_pred)

print(f"\nDumb baseline accuracy (predict 0 for everyone): {dumb_acc:.3f}")
print(f"Our model accuracy:                              {our_acc:.3f}")
print(f"Real improvement:                               +{our_acc - dumb_acc:.3f}")

## The Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print()

tn, fp, fn, tp = cm.ravel()
print(f"TN (correctly said not survived): {tn}")
print(f"FP (wrongly said survived):       {fp}  ← false alarm")
print(f"FN (missed actual survivors):     {fn}  ← missed case")
print(f"TP (correctly said survived):     {tp}")

In [ ]:
# Visualise as heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not survived', 'Survived'],
            yticklabels=['Not survived', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Titanic Survival')
plt.tight_layout()
plt.show()

## Accuracy from the Confusion Matrix

Accuracy = (TP + TN) / (TP + TN + FP + FN) — just the diagonal divided by everything.

In [ ]:
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
print(f"Accuracy from confusion matrix: {accuracy_manual:.3f}")
print(f"Accuracy from sklearn:          {our_acc:.3f}")
print("Same number — accuracy is just the diagonal of the confusion matrix.")

## Key Insight

The confusion matrix shows WHERE your model makes mistakes, not just how many:

- Large FN = your model misses real positive cases (bad for cancer detection)
- Large FP = your model raises too many false alarms (bad for spam filters)
- The diagonal (TN + TP) = correct predictions
- Everything off the diagonal = mistakes

> Next: 3.2 Precision & Recall — metrics built directly from these 4 numbers.

## Practice Task

In [ ]:
# Q1 — Print the confusion matrix and label each value (TN, FP, FN, TP)
# YOUR CODE HERE

# Q2 — Calculate accuracy manually from the confusion matrix values
# YOUR CODE HERE

# Q3 — How many survivors did the model MISS (FN)?
#       Is that number acceptable? Write your answer as a comment.
# ANSWER:

# Q4 — For Titanic survival prediction, which is worse — FP or FN? Why?
# ANSWER: